# D+006 종합 실습 — 주문 데이터 전처리 파이프라인 + 품질 리포트

**목표**: 원본 주문 CSV를 `preprocess()` 한 줄로 정제하고, 파이프라인이 *어떤 행을 얼마나 걸러냈는지* 근거를 남기는 품질 리포트를 만든다.

| 구성 | 내용 |
|---|---|
| 시나리오 0 | 원본 CSV 준비 |
| 시나리오 1 | 단계 함수 6개 (함수당 한 책임) |
| 시나리오 2 | `preprocess()` end-to-end 파이프라인 |
| 시나리오 3 | CSV 저장 + 재로드 shape 검증 |
| 시나리오 4 | **품질 리포트 함수** `quality_report()` (D+009 재사용) |

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

원본 데이터 준비
실무에서는 외부(DB·API·파일)에서 받아오는 단계입니다. 여기서는 **현실적인 오류**를 의도적으로 섞은 주문 데이터 500건을 생성합니다.

- 표기 혼재: `" 서울"`, `"Seoul"`, `"GOLD"`, `" Online "`
- 결측: `amount`, `customer_age` 일부 NaN
- 불가능한 값: 나이 3세·150세, 수량 99개
- 완전 중복 행

In [2]:
rng = np.random.default_rng(42)
N = 500

orders = pd.DataFrame({
    "order_id":     np.arange(1, N + 1),
    "region":       rng.choice(["서울", " 서울", "Seoul", "부산", "부산 ", "대구", "광주"], N),
    "membership":   rng.choice(["basic", "Basic", "silver", "SILVER", "gold", "Gold", "vip", "VIP"], N),
    "channel":      rng.choice(["online", " Online ", "app", "APP", "offline"], N),
    "category":     rng.choice(["fashion", "food", "electronics", "beauty"], N),
    "customer_age": rng.integers(15, 70, N).astype(float),
    "quantity":     rng.integers(1, 8, N),
    "amount":       rng.lognormal(mean=10.3, sigma=0.8, size=N).round(-2),
})

# 오류 주입
orders.loc[rng.choice(N, 25, replace=False), "amount"] = np.nan
orders.loc[rng.choice(N, 15, replace=False), "customer_age"] = np.nan
orders.loc[rng.choice(N, 8,  replace=False), "customer_age"] = rng.choice([3, 5, 120, 150], 8)
orders.loc[rng.choice(N, 6,  replace=False), "quantity"] = rng.choice([25, 50, 99], 6)

# 완전 중복 행 10개 추가
orders = pd.concat([orders, orders.sample(10, random_state=1)], ignore_index=True)

print("원본 shape:", orders.shape)
orders.head()

원본 shape: (510, 8)


,order_id,region,membership,channel,category,customer_age,quantity,amount
0,1,서울,basic,offline,electronics,63.0,5,64200.0
1,2,대구,Gold,APP,food,18.0,1,16700.0
2,3,부산,VIP,app,fashion,36.0,6,15000.0
3,4,부산,vip,offline,food,40.0,1,42900.0
4,5,부산,silver,online,electronics,58.0,1,65600.0


In [3]:
# ─────────────────────────────────────────────
# 시나리오 0 — 원본 CSV 파일 준비
# ─────────────────────────────────────────────
work_dir = Path("d006_work")
work_dir.mkdir(exist_ok=True)
input_csv  = work_dir / "orders_raw.csv"
output_csv = work_dir / "orders_clean.csv"

orders.to_csv(input_csv, index=False)
print("원본 CSV 저장:", input_csv, "—", input_csv.stat().st_size, "bytes")

원본 CSV 저장: d006_work/orders_raw.csv — 23097 bytes


단계 함수 (한 함수 = 한 책임)

| 함수 | 책임 |
|---|---|
| `load_orders` | 읽기 |
| `clean_strings_full` | 문자열 표기 통일 |
| `drop_invalid_rows` | 결측·불가능 값·중복 제거 |
| `add_derived` | 파생 컬럼 생성 |
| `encode_full` | Ordinal + One-Hot 인코딩 |
| `scale_with_robust` | Robust 스케일링 |

In [4]:
def load_orders(path):
    """CSV에서 주문 데이터를 읽어 옵니다."""
    return pd.read_csv(path)

def clean_strings_full(df):
    """문자열 컬럼의 공백·대소문자·표기 혼재를 정리합니다."""
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )

def drop_invalid_rows(df, age_min=10, age_max=80, qty_max=10):
    """불가능한 값과 결측을 제거합니다."""
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
        .drop_duplicates()
        .reset_index(drop=True)
    )

def add_derived(df):
    """분석용 파생 컬럼을 추가합니다."""
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
        amount_class=lambda d: np.where(d["amount"] >= 100_000, "high",
                                np.where(d["amount"] >= 30_000, "mid", "low"))
    )

def encode_full(df):
    """membership=Ordinal, 나머지=One-Hot."""
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    one_hots = [
        pd.get_dummies(out["region"],   prefix="region", dtype=int),
        pd.get_dummies(out["channel"],  prefix="ch",     dtype=int),
        pd.get_dummies(out["category"], prefix="cat",    dtype=int),
    ]
    return pd.concat([out] + one_hots, axis=1)

def scale_with_robust(df, cols=("customer_age", "amount", "quantity")):
    """수치형 컬럼을 Robust로 스케일링하고 _scaled 접미사를 추가합니다."""
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled,
                             columns=[f"{c}_scaled" for c in cols],
                             index=df.index)
    return pd.concat([df, scaled_df], axis=1)

print("단계 함수 6개 정의 완료.")

단계 함수 6개 정의 완료.


end-to-end 파이프라인 (`preprocess()` 한 줄 호출)

In [5]:
def preprocess(input_path):
    """원본 CSV 경로를 받아 전처리된 DataFrame을 반환합니다."""
    return (
        load_orders(input_path)
        .pipe(clean_strings_full)
        .pipe(drop_invalid_rows, age_min=10, age_max=80, qty_max=10)
        .pipe(add_derived)
        .pipe(encode_full)
        .pipe(scale_with_robust)
    )

clean_df = preprocess(input_csv)          # ← 진짜 한 줄 호출
print("입력 행 수 → 출력 행 수:", orders.shape[0], "→", clean_df.shape[0])
print("출력 컬럼 수:", clean_df.shape[1])
clean_df.head(3)

입력 행 수 → 출력 행 수: 510 → 449
출력 컬럼 수: 26


,order_id,region,membership,channel,category,customer_age,quantity,amount,amount_log,is_premium,amount_class,membership_ord,region_광주,region_대구,region_부산,region_서울,ch_app,ch_offline,ch_online,cat_beauty,cat_electronics,cat_fashion,cat_food,customer_age_scaled,amount_scaled,quantity_scaled
0,1,서울,basic,offline,electronics,63.0,5,64200.0,11.069774,0,mid,1,0,0,0,1,0,1,0,0,1,0,0,0.884615,1.241042,0.25
1,2,대구,gold,app,food,18.0,1,16700.0,9.723224,1,low,3,0,1,0,0,1,0,0,0,0,0,1,-0.846154,-0.306189,-0.75
2,3,부산,vip,app,fashion,36.0,6,15000.0,9.615872,1,low,4,0,0,1,0,1,0,0,0,0,1,0,-0.153846,-0.361564,0.50


저장 + 재로드 검증

In [6]:
clean_df.to_csv(output_csv, index=False)

reloaded = pd.read_csv(output_csv)
print("저장 파일:", output_csv, "—", output_csv.stat().st_size, "bytes")
print("저장 직전 shape:", clean_df.shape)
print("다시 읽은  shape:", reloaded.shape)
print("동일한가?:", clean_df.shape == reloaded.shape)
assert clean_df.shape == reloaded.shape, "저장/재로드 후 shape 불일치!"

저장 파일: d006_work/orders_clean.csv — 61789 bytes
저장 직전 shape: (449, 26)
다시 읽은  shape: (449, 26)
동일한가?: True


품질 리포트 함수 

> 데이터 분석가의 무기는 결과만이 아니라 **결정의 근거**다.

설계 원칙
- **입력(raw)과 출력(clean) 두 DataFrame만 받는다** → 어떤 파이프라인에도 재사용 가능 (D+009에서 그대로 사용)
- 화면 출력과 함께 **dict를 반환**한다 → 다음 모듈에서 수치를 표·그래프로 재활용
- 행 제거 사유를 단계별로 추적하는 `trace_drop_steps()`는 **별도 함수**로 분리 (한 함수 = 한 책임)

| 리포트 항목 | 내용 |
|---|---|
| ① 행 수 | 입력 / 출력 / 제거 수 / 제거 비율 |
| ② 결측 | 컬럼별 결측 비율 상위 5개 (입력 vs 출력) |
| ③ 컬럼 | 새로 만들어진 파생·인코딩 컬럼 수와 목록 |
| ④ 자료형 | dtype 분포 요약 |
| (+) 제거 사유 | 필터 단계별 제거 행 수 |

In [7]:
def trace_drop_steps(df, age_min=10, age_max=80, qty_max=10):
    """drop_invalid_rows의 각 필터가 몇 행을 제거했는지 단계별로 추적합니다."""
    steps = [
        ("결측 제거 (amount, customer_age)", lambda d: d.dropna(subset=["amount", "customer_age"])),
        (f"나이 범위 ({age_min} < age < {age_max})", lambda d: d[(d["customer_age"] > age_min) & (d["customer_age"] < age_max)]),
        (f"수량 상한 (quantity <= {qty_max})",       lambda d: d[d["quantity"] <= qty_max]),
        ("완전 중복 제거",                           lambda d: d.drop_duplicates()),
    ]
    rows, current = [], df
    for name, fn in steps:
        before = len(current)
        current = fn(current)
        rows.append({"단계": name, "이전": before, "이후": len(current),
                     "제거": before - len(current)})
    log = pd.DataFrame(rows)
    log["제거 비율(%)"] = (log["제거"] / len(df) * 100).round(2)
    return log


def quality_report(raw_df, clean_df, top_n=5, drop_log=None, verbose=True):
    """
    전처리 전/후 DataFrame을 비교해 품질 리포트를 출력하고 dict로 반환합니다.

    Parameters
    ----------
    raw_df   : 파이프라인 입력 DataFrame
    clean_df : 파이프라인 출력 DataFrame
    top_n    : 결측 비율 상위 몇 개를 보여줄지
    drop_log : trace_drop_steps() 결과 (선택)
    verbose  : True면 화면에 출력
    """
    # ① 행 수
    n_in, n_out = len(raw_df), len(clean_df)
    n_removed   = n_in - n_out
    removed_pct = n_removed / n_in * 100 if n_in else 0.0

    # ② 결측 비율 상위 N
    miss_in  = raw_df.isna().mean().mul(100).sort_values(ascending=False)
    miss_out = clean_df.isna().mean().mul(100).sort_values(ascending=False)
    missing_top = pd.DataFrame({
        "입력 결측(%)": miss_in.head(top_n).round(2),
        "출력 결측(%)": miss_out.reindex(miss_in.head(top_n).index).round(2),
    })
    missing_top.index.name = "컬럼"

    # ③ 새로 만들어진 컬럼 (파생 / 인코딩 / 스케일링으로 분류)
    new_cols = [c for c in clean_df.columns if c not in raw_df.columns]
    def _kind(c):
        if c.endswith("_scaled"):                         return "스케일링"
        if c.endswith("_ord"):                            return "인코딩(Ordinal)"
        if c.startswith(("region_", "ch_", "cat_")):      return "인코딩(One-Hot)"
        return "파생"
    new_col_summary = pd.Series([_kind(c) for c in new_cols], index=new_cols)
    kind_counts = new_col_summary.value_counts()

    # ④ 자료형 분포
    dtype_dist = pd.DataFrame({
        "입력": raw_df.dtypes.astype(str).value_counts(),
        "출력": clean_df.dtypes.astype(str).value_counts(),
    }).fillna(0).astype(int)
    dtype_dist.index.name = "dtype"

    report = {
        "rows_in": n_in, "rows_out": n_out,
        "rows_removed": n_removed, "removed_pct": round(removed_pct, 2),
        "missing_top": missing_top,
        "n_cols_in": raw_df.shape[1], "n_cols_out": clean_df.shape[1],
        "n_new_cols": len(new_cols), "new_cols_by_kind": kind_counts.to_dict(),
        "new_cols": new_cols, "dtype_dist": dtype_dist, "drop_log": drop_log,
    }

    if verbose:
        line = "═" * 58
        print(line); print("  📋 데이터 품질 리포트"); print(line)

        print("\n① 행 수")
        print(f"   입력 행 수   : {n_in:>6,}")
        print(f"   출력 행 수   : {n_out:>6,}")
        print(f"   제거된 행 수 : {n_removed:>6,}  ({removed_pct:.2f}%)")

        if drop_log is not None:
            print("\n   ▸ 제거 사유 (단계별)")
            print(drop_log.to_string(index=False).replace("\n", "\n   ").join(["   ", ""]))

        print(f"\n② 컬럼별 결측 비율 (입력 기준 상위 {top_n})")
        print(missing_top.to_string().replace("\n", "\n   ").join(["   ", ""]))

        print("\n③ 새로 만들어진 컬럼")
        print(f"   컬럼 수 : {raw_df.shape[1]} → {clean_df.shape[1]}  (+{len(new_cols)})")
        for kind, cnt in kind_counts.items():
            cols = new_col_summary[new_col_summary == kind].index.tolist()
            print(f"   - {kind:<14}: {cnt:>2}개  {cols}")

        print("\n④ 자료형 분포")
        print(dtype_dist.to_string().replace("\n", "\n   ").join(["   ", ""]))
        print(line)

    return report

### 리포트 실행

In [8]:
raw_df   = load_orders(input_csv)
drop_log = trace_drop_steps(clean_strings_full(raw_df), age_min=10, age_max=80, qty_max=10)

report = quality_report(raw_df, clean_df, top_n=5, drop_log=drop_log)

══════════════════════════════════════════════════════════
  📋 데이터 품질 리포트
══════════════════════════════════════════════════════════

① 행 수
   입력 행 수   :    510
   출력 행 수   :    449
   제거된 행 수 :     61  (11.96%)

   ▸ 제거 사유 (단계별)
                             단계  이전  이후  제거  제거 비율(%)
   결측 제거 (amount, customer_age) 510 468  42      8.24
          나이 범위 (10 < age < 80) 468 461   7      1.37
         수량 상한 (quantity <= 10) 461 457   4      0.78
                       완전 중복 제거 457 449   8      1.57

② 컬럼별 결측 비율 (입력 기준 상위 5)
                 입력 결측(%)  출력 결측(%)
   컬럼                              
   amount            5.29       0.0
   customer_age      2.94       0.0
   order_id          0.00       0.0
   region            0.00       0.0
   membership        0.00       0.0

③ 새로 만들어진 컬럼
   컬럼 수 : 8 → 26  (+18)
   - 인코딩(One-Hot)  : 11개  ['region_광주', 'region_대구', 'region_부산', 'region_서울', 'ch_app', 'ch_offline', 'ch_online', 'cat_beauty', 'cat_electronics', 'cat_fashion', 'cat_food']
   - 파생 

### 검증 — 리포트 수치가 파이프라인과 일치하는가?

In [9]:
# 단계별 제거 합계 == 전체 제거 행 수
assert drop_log["제거"].sum() == report["rows_removed"], "단계별 합계 불일치"
# 단계 추적의 최종 행 수 == 파이프라인 출력 행 수
assert drop_log["이후"].iloc[-1] == report["rows_out"], "최종 행 수 불일치"
# 저장 파일도 동일 shape
assert reloaded.shape == clean_df.shape

print("✅ 모든 검증 통과")
print("반환 dict 키:", list(report.keys()))

✅ 모든 검증 통과
반환 dict 키: ['rows_in', 'rows_out', 'rows_removed', 'removed_pct', 'missing_top', 'n_cols_in', 'n_cols_out', 'n_new_cols', 'new_cols_by_kind', 'new_cols', 'dtype_dist', 'drop_log']


회고 — 결정의 근거

- 나이 범위 `10 < age < 80`, 수량 상한 `10`은 **도메인 가정**입니다. 리포트의 단계별 제거 표 덕분에 "이 기준 때문에 몇 %가 빠졌는가"를 설명할 수 있습니다.
- 출력 결측이 0%가 된 것은 `dropna`의 결과이므로, 결측이 많은 데이터라면 **삭제 대신 대체(imputation)** 를 검토해야 합니다.
- `quality_report()`는 입력·출력 DataFrame만 받고 dict를 반환하므로, **D+009 종합 프로젝트**에서 다른 파이프라인에도 그대로 붙일 수 있습니다.